# ML decoder: CPU + GPU check on a Kaggle T4

Settings: Accelerator = GPU T4, Internet = On. Runs the hand-in in `submission_ml/` exactly as `run_local.py` does. Default mode is `exact`: every shot is decoded on the GPU with the full tables, no pruning.
The first import builds the tables (about 20-30 minutes on Kaggle's 4 CPU cores); later imports load them from `mld_cache/`.

In [ ]:
!nvidia-smi -L
!nproc && free -g | head -2
!git clone -q -b dev/coset-walk https://github.com/sau22rane/QEC_Decoder_Hakathon_2026_ku.git repo
%cd repo
!pip install -q stim==1.16.0 numpy==2.5.4  # numpy 2.1.3 drops 3 rows in load_kaggle_csv
!pip install -q --no-deps -e .
import torch, numba, numpy; print(torch.__version__, torch.cuda.is_available(), numba.__version__, numpy.__version__)

## 1. Labelled set: LER, timing, determinism (exact mode, every shot on the GPU)

In [ ]:
!python submission_template/run_local.py --decoder submission_ml --problem-set data/train --runs 3 --threads 4 --out submission_train_ml_gpu.csv 2>&1 | grep -v -i warn

## 2. Test set: write the submission file and compare with the Mac (Apple GPU) predictions

Zero differing bits means the output is independent of the GPU model (bits near 0.5 are recomputed in float64).

In [ ]:
!python submission_template/run_local.py --decoder submission_ml --problem-set data/test --runs 1 --threads 4 --out submission_test_ml_gpu.csv 2>&1 | grep -v -i warn
import numpy as np
a = np.loadtxt('submission_test_ml_gpu.csv', delimiter=',', skiprows=1, usecols=(1, 2, 3, 4), dtype=np.uint8)
ref = np.unpackbits(np.load('submission_ml/reference_test_pred_mps.npy'))[:a.size].reshape(a.shape)
print('bits differing from the Mac reference:', int((a != ref).sum()), 'of', a.size)

## 3. Metrics for later estimates

Everything is collected in `metrics` and written to `mld_metrics.json` (download it from the Output tab).
- hardware and library versions
- table load time and GPU memory
- raw float32 matrix-product speed of the GPU (TFLOPS) and dense-pass shots/s at several batch sizes
- CPU pass speed at 1, 2 and 4 threads for both profiles (core-seconds per 200k shots)
- per-stage seconds and shot counts of a full exact `decode()` on `data/train` (distinct syndromes, shared left/right parts, float64 ties), plus LER per basis
- how the share of shots sent to the GPU, and the LER, change with TAU

In [ ]:
import json, os, platform, subprocess, sys, time
import numpy as np, torch, numba, stim
sys.path[:0] = ['submission_ml', 'src']  # 'src': the editable install isn't visible to an already running kernel
t0 = time.time(); import decoder, mlddense, mldfast; import_seconds = time.time() - t0
from hackathon_2026 import load_kaggle_csv
metrics = dict(import_seconds=import_seconds, python=platform.python_version(), torch=torch.__version__, numba=numba.__version__,
               numpy=np.__version__, cuda=torch.version.cuda, gpu=torch.cuda.get_device_name(0), cpus=os.cpu_count(),
               cpu_model=subprocess.run("lscpu | grep 'Model name'", shell=True, capture_output=True, text=True).stdout.strip(),
               numba_threads=numba.get_num_threads(), threading_layer=numba.threading_layer(),
               gpu_memory_gb_after_import=torch.cuda.memory_allocated() / 2**30)
data = load_kaggle_csv('data/train/train.csv')
circ = {b: stim.Circuit.from_file(f'submission_ml/{b}_circuit.stim') for b in 'XZ'}
dets = {b: data['detector_events'][data['basis'] == b] for b in 'XZ'}
obs = {b: data['observable_flips'][data['basis'] == b] for b in 'XZ'}

# raw GPU float32 matrix-product speed
a = torch.rand(4096, 16384, device='cuda'); m = torch.rand(16384, 16384, device='cuda')
(a @ m); torch.cuda.synchronize(); t = time.time()
for _ in range(5): (a @ m)
torch.cuda.synchronize(); metrics['gpu_matmul_tflops_fp32'] = 5 * 2 * 4096 * 16384**2 / (time.time() - t) / 1e12
del a, m

# dense pass throughput by batch size
key, tb = decoder._tables(circ['X']); dt = decoder._dense(circ['X'], key, tb)
metrics['dense_shots_per_s'] = {}
for batch in (256, 1024, 4096):
    d = dets['X'][:4 * batch]; mlddense.run_all(dt, d[:batch], batch); torch.cuda.synchronize()
    t = time.time(); mlddense.run_all(dt, d, batch); torch.cuda.synchronize()
    metrics['dense_shots_per_s'][batch] = len(d) / (time.time() - t)
metrics['gpu_memory_gb_peak'] = torch.cuda.max_memory_allocated() / 2**30

# CPU pass speed by thread count and profile
metrics['cpu_pass'] = []
for threads in (1, 2, 4):
    if threads > numba.config.NUMBA_NUM_THREADS: continue
    numba.set_num_threads(threads)
    for prof, nshots in (('fast', 20000), ('max', 4000)):
        d = dets['X'][:nshots]; t = time.time(); mldfast.run(tb, d, *decoder.PROFILES[prof]); sec = time.time() - t
        metrics['cpu_pass'].append(dict(threads=threads, profile=prof, shots=nshots, seconds=sec,
                                        core_seconds_per_200k=sec * threads * 200000 / nshots))
numba.set_num_threads(numba.config.NUMBA_NUM_THREADS)

# full decode: per-stage statistics and LER
decoder.STATS.clear(); metrics['decode'] = {}
for b in 'XZ':
    pred = decoder.decode(circ[b], dets[b])
    metrics['decode'][b] = dict(decoder.STATS[-1], ler=float(np.mean(pred.astype(bool) != obs[b])))
metrics['ler_train'] = float(np.mean([metrics['decode'][b]['ler'] for b in 'XZ']))

# hybrid mode (CPU pass + GPU on uncertain shots): effect of TAU on the first 20k shots per basis
decoder.MODE = 'hybrid'; metrics['tau_sweep'] = []
for tau in (0.1, 0.2, 0.3, 0.4, 0.45, 0.5):
    decoder.TAU = tau; decoder.STATS.clear(); wrong = []
    for b in 'XZ':
        wrong.append(decoder.decode(circ[b], dets[b][:20000]).astype(bool) != obs[b][:20000])
    metrics['tau_sweep'].append(dict(tau=tau, gpu_share=sum(s['gpu_shots'] for s in decoder.STATS) / 40000,
                                     ler=float(np.mean(wrong)), gpu_seconds=sum(s['gpu_seconds'] for s in decoder.STATS),
                                     changed_bits=sum(s['gpu_changed_bits'] for s in decoder.STATS)))
decoder.TAU = 0.3; decoder.MODE = 'exact'
json.dump(metrics, open('/kaggle/working/mld_metrics.json', 'w'), indent=1, default=float)
print(json.dumps(metrics, indent=1, default=float))